# 🧠 Módulo 4 — Deep Learning: CNNs, RNNs y Aplicaciones
**Diplomado Superior en Redes Neuronales Artificiales y Deep Learning**  
Machote de práctica — Diana Blanco  

---
### Temario cubierto:
1. Fundamentos e introducción a redes neuronales  
2. Arquitecturas de aprendizaje profundo (CNNs y RNNs)  
3. Entrenamiento: backprop, descenso por gradiente, hiperparámetros  
4. Aplicaciones: Visión por Computadora y PLN  
5. Plataformas comerciales para IA  

> 📌 **Instrucciones de uso:** Cada sección tiene celdas con código base y comentarios explicativos.  
> Las celdas marcadas con `# TODO` son para que completes tú según el dataset del ejercicio.  
> Las celdas marcadas con `# MACHOTE` ya funcionan y puedes reusar/adaptar.

---
## ⚙️ 0. Imports y configuración general

In [ ]:
# MACHOTE — Imports estándar para todo el módulo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

# Utilidades de sklearn (preprocesamiento, métricas)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix

# Reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

# Verificar GPU disponible
print("GPUs disponibles:", tf.config.list_physical_devices('GPU'))
print("TensorFlow versión:", tf.__version__)

---
## 📚 1. Fundamentos de Redes Neuronales

### ¿Qué es una red neuronal artificial?
Es un modelo computacional inspirado en el cerebro biológico.  
Está formada por **capas de neuronas** que transforman los datos de entrada hasta producir una salida (predicción).

```
Entrada → [Capas ocultas] → Salida
   X    →   (pesos W)    →   ŷ
```

### Componentes clave:
| Componente | Descripción |
|---|---|
| **Neurona** | Unidad básica: suma ponderada + función de activación |
| **Capa densa** | Cada neurona conectada con todas las de la capa anterior |
| **Función de activación** | Introduce no-linealidad (ReLU, Sigmoid, Softmax) |
| **Función de pérdida** | Mide qué tan equivocado está el modelo |
| **Optimizador** | Actualiza los pesos para reducir el error (SGD, Adam) |

In [ ]:
# MACHOTE — Red neuronal densa básica (Fully Connected / MLP)
# Útil para datos tabulares o como baseline antes de CNN/RNN

def crear_mlp(input_shape, num_clases, capas_ocultas=[128, 64], dropout=0.3):
    """
    Crea un Perceptrón Multicapa configurable.
    
    Parámetros:
    - input_shape: forma de los datos de entrada (ej. (784,) para MNIST aplanado)
    - num_clases: número de categorías de salida
    - capas_ocultas: lista con número de neuronas por capa oculta
    - dropout: fracción de neuronas a desactivar aleatoriamente (regularización)
    """
    modelo = keras.Sequential(name="MLP_Basico")
    
    # Capa de entrada
    modelo.add(layers.Input(shape=input_shape))
    
    # Capas ocultas
    for neuronas in capas_ocultas:
        modelo.add(layers.Dense(neuronas, activation='relu'))  # ReLU = max(0, x)
        modelo.add(layers.BatchNormalization())                 # Normaliza activaciones → más estable
        modelo.add(layers.Dropout(dropout))                    # Regularización → evita overfitting
    
    # Capa de salida
    if num_clases == 2:
        # Clasificación binaria → Sigmoid (da probabilidad 0-1)
        modelo.add(layers.Dense(1, activation='sigmoid'))
    else:
        # Clasificación multiclase → Softmax (da probabilidades que suman 1)
        modelo.add(layers.Dense(num_clases, activation='softmax'))
    
    return modelo

# Ejemplo de uso
# TODO: ajusta input_shape y num_clases según tu dataset
mlp = crear_mlp(input_shape=(20,), num_clases=3)
mlp.summary()

---
## 🔁 2. Entrenamiento: Backpropagation y Descenso por Gradiente

### ¿Cómo aprende la red?
1. **Forward pass:** los datos pasan por la red y se obtiene una predicción ŷ  
2. **Cálculo de pérdida:** se compara ŷ con y_real usando la función de pérdida  
3. **Backward pass (backprop):** se calculan los gradientes ∂Loss/∂W para cada peso  
4. **Actualización:** el optimizador mueve los pesos en dirección contraria al gradiente  

```
W_nuevo = W_viejo - learning_rate × gradiente
```

### Hiperparámetros principales:
| Hiperparámetro | Qué controla | Valor típico |
|---|---|---|
| `learning_rate` | Tamaño de cada paso de actualización | 0.001 (Adam) |
| `batch_size` | Muestras por actualización de pesos | 32, 64, 128 |
| `epochs` | Vueltas completas sobre el dataset | 10–100 |
| `dropout` | % neuronas desactivadas aleatoriamente | 0.2–0.5 |
| `n_layers` / `n_units` | Profundidad y ancho de la red | depende del problema |

In [ ]:
# MACHOTE — Compilar y entrenar cualquier modelo Keras

def compilar_y_entrenar(modelo, X_train, y_train, X_val, y_val,
                        num_clases=3, lr=0.001, epochs=30, batch_size=32):
    """
    Compila el modelo con la función de pérdida correcta según el número de clases,
    y lo entrena con early stopping para evitar overfitting.
    """
    # Seleccionar función de pérdida automáticamente
    if num_clases == 2:
        loss = 'binary_crossentropy'
        metrics_list = ['accuracy']
    else:
        # sparse = las etiquetas son enteros (0, 1, 2...)
        # categorical = las etiquetas ya están en One-Hot
        loss = 'sparse_categorical_crossentropy'  # cambiar si ya hiciste OHE
        metrics_list = ['accuracy']
    
    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss=loss,
        metrics=metrics_list
    )
    
    # Callbacks útiles
    cb_early_stop = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,           # detiene si no mejora en 5 epochs seguidas
        restore_best_weights=True  # regresa al mejor checkpoint automáticamente
    )
    cb_reduce_lr = callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,           # reduce lr a la mitad
        patience=3,
        verbose=1
    )
    
    history = modelo.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[cb_early_stop, cb_reduce_lr],
        verbose=1
    )
    
    return history

# TODO: llama esta función con tu modelo y tus datos
# history = compilar_y_entrenar(mlp, X_train, y_train, X_val, y_val, num_clases=3)

In [ ]:
# MACHOTE — Graficar curvas de entrenamiento (loss y accuracy)

def graficar_historia(history, titulo="Entrenamiento"):
    """
    Muestra las curvas de pérdida y accuracy para train y validación.
    Útil para detectar overfitting (train mejora pero val se estanca).
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(titulo, fontsize=14, fontweight='bold')
    
    # Pérdida
    axes[0].plot(history.history['loss'], label='Train Loss', color='steelblue')
    axes[0].plot(history.history['val_loss'], label='Val Loss', color='tomato', linestyle='--')
    axes[0].set_title('Función de Pérdida')
    axes[0].set_xlabel('Época')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Accuracy
    axes[1].plot(history.history['accuracy'], label='Train Acc', color='steelblue')
    axes[1].plot(history.history['val_accuracy'], label='Val Acc', color='tomato', linestyle='--')
    axes[1].set_title('Exactitud (Accuracy)')
    axes[1].set_xlabel('Época')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# TODO: descomenta cuando tengas history
# graficar_historia(history, "MLP — Dataset X")

---
## 🖼️ 3. Redes Convolucionales (CNNs) — Visión por Computadora

### ¿Por qué no usar MLP para imágenes?
Una imagen de 224×224×3 tiene ~150k píxeles → demasiados pesos en una capa densa.  
Las CNNs resuelven esto con **convoluciones**: filtros pequeños que detectan patrones locales.

### Capas típicas de una CNN:
| Capa | Función |
|---|---|
| `Conv2D` | Aplica filtros para detectar bordes, texturas, formas |
| `MaxPooling2D` | Reduce el tamaño espacial (downsample), retiene lo importante |
| `BatchNormalization` | Estabiliza el entrenamiento |
| `Flatten` | Convierte el mapa de features en vector 1D |
| `Dense` | Capas FC finales para clasificar |

In [ ]:
# MACHOTE — CNN para clasificación de imágenes

def crear_cnn(input_shape, num_clases):
    """
    CNN estándar para clasificación de imágenes.
    
    input_shape: (alto, ancho, canales) — ej. (28, 28, 1) para MNIST en escala de grises
                                          ej. (32, 32, 3) para CIFAR-10 en RGB
    num_clases:  número de categorías
    """
    modelo = keras.Sequential(name="CNN_Clasificacion")
    
    # --- Bloque 1: detectar features de bajo nivel (bordes, colores) ---
    modelo.add(layers.Conv2D(32, (3,3), activation='relu',
                             padding='same', input_shape=input_shape))
    # padding='same' → mantiene dimensiones espaciales
    modelo.add(layers.BatchNormalization())
    modelo.add(layers.MaxPooling2D((2,2)))   # reduce a la mitad: 28→14
    
    # --- Bloque 2: features de nivel medio ---
    modelo.add(layers.Conv2D(64, (3,3), activation='relu', padding='same'))
    modelo.add(layers.BatchNormalization())
    modelo.add(layers.MaxPooling2D((2,2)))   # 14→7
    modelo.add(layers.Dropout(0.25))
    
    # --- Bloque 3: features de alto nivel ---
    modelo.add(layers.Conv2D(128, (3,3), activation='relu', padding='same'))
    modelo.add(layers.BatchNormalization())
    modelo.add(layers.GlobalAveragePooling2D())  # alternativa a Flatten, menos parámetros
    
    # --- Clasificador final ---
    modelo.add(layers.Dense(128, activation='relu'))
    modelo.add(layers.Dropout(0.5))
    
    if num_clases == 2:
        modelo.add(layers.Dense(1, activation='sigmoid'))
    else:
        modelo.add(layers.Dense(num_clases, activation='softmax'))
    
    return modelo

# Ejemplo con MNIST (dígitos escritos a mano)
# TODO: cambia input_shape según tu dataset de imágenes
cnn = crear_cnn(input_shape=(28, 28, 1), num_clases=10)
cnn.summary()

In [ ]:
# MACHOTE — Cargar y preprocesar MNIST para probar la CNN

# Cargar dataset incluido en Keras
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = keras.datasets.mnist.load_data()

# Normalizar píxeles a rango [0, 1]
X_train_cnn = X_train_raw.astype('float32') / 255.0
X_test_cnn  = X_test_raw.astype('float32') / 255.0

# Agregar canal de color: (60000, 28, 28) → (60000, 28, 28, 1)
# Las CNNs esperan 4D: (muestras, alto, ancho, canales)
X_train_cnn = X_train_cnn[..., np.newaxis]
X_test_cnn  = X_test_cnn[..., np.newaxis]

# Separar validación del train
X_train_cnn, X_val_cnn, y_train_cnn, y_val_cnn = train_test_split(
    X_train_cnn, y_train_raw, test_size=0.1, random_state=42
)

print(f"Train: {X_train_cnn.shape} | Val: {X_val_cnn.shape} | Test: {X_test_cnn.shape}")
print(f"Clases: {np.unique(y_train_cnn)}")

# Visualizar algunas muestras
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train_cnn[i].squeeze(), cmap='gray')
    ax.set_title(f"Clase: {y_train_cnn[i]}")
    ax.axis('off')
plt.suptitle("Muestras MNIST", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# MACHOTE — Data Augmentation (aumentar datos de imagen artificialmente)
# Muy útil cuando tienes pocos datos de entrenamiento

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),         # voltear imagen horizontalmente
    layers.RandomRotation(0.1),              # rotar ±10%
    layers.RandomZoom(0.1),                  # zoom ±10%
    layers.RandomTranslation(0.1, 0.1),      # desplazar ±10%
], name="data_augmentation")

# Para usar dentro de un modelo funcional:
# inputs = keras.Input(shape=(32, 32, 3))
# x = data_augmentation(inputs)  # solo se aplica en entrenamiento, no en eval
# x = crear_cnn_funcional(x)
# ...
print("Data augmentation configurado ✓")

In [ ]:
# MACHOTE — Transfer Learning con modelo preentrenado
# Reusar pesos entrenados en ImageNet para clasificar tus propias imágenes
# MUY útil cuando tienes pocos datos

def crear_transfer_learning(num_clases, input_shape=(224, 224, 3), base='MobileNetV2'):
    """
    Crea un modelo de transfer learning con base preentrenada.
    Opciones de base: 'MobileNetV2', 'VGG16', 'ResNet50', 'EfficientNetB0'
    """
    # Cargar base sin la capa de clasificación final (include_top=False)
    if base == 'MobileNetV2':
        base_model = keras.applications.MobileNetV2(
            input_shape=input_shape, include_top=False, weights='imagenet'
        )
    elif base == 'VGG16':
        base_model = keras.applications.VGG16(
            input_shape=input_shape, include_top=False, weights='imagenet'
        )
    
    # Congelar la base: sus pesos NO se actualizan durante el primer entrenamiento
    base_model.trainable = False
    
    # Construir sobre la base
    inputs = keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)   # training=False → BatchNorm en modo eval
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    
    if num_clases == 2:
        outputs = layers.Dense(1, activation='sigmoid')(x)
    else:
        outputs = layers.Dense(num_clases, activation='softmax')(x)
    
    modelo = keras.Model(inputs, outputs, name=f"TransferLearning_{base}")
    return modelo, base_model

# TODO: descomenta para usar
# modelo_tl, base = crear_transfer_learning(num_clases=5)
# modelo_tl.summary()

# FINE-TUNING (después del primer entrenamiento):
# base.trainable = True
# Recompila con lr MUY bajo (1e-5) y vuelve a entrenar pocas epochs
print("Transfer Learning machote listo ✓")

---
## 📝 4. Redes Recurrentes (RNNs / LSTMs) — PLN y Series de Tiempo

### ¿Por qué RNN?
Las MLPs y CNNs no tienen memoria — cada entrada se procesa de forma independiente.  
Las RNNs sí tienen **estado oculto** que se pasa de paso en paso, ideal para:
- Texto (secuencias de palabras)
- Series de tiempo (temperatura, ventas, señales)
- Audio

### Problema de las RNNs simples:
**Vanishing gradient** — el gradiente se desvanece con secuencias largas.  
**Solución:** LSTM (Long Short-Term Memory) y GRU (Gated Recurrent Unit)

| Arquitectura | Memoria | Velocidad | Uso recomendado |
|---|---|---|---|
| `SimpleRNN` | Corta | Rápida | Secuencias muy cortas |
| `LSTM` | Larga | Lenta | Texto, series de tiempo largas |
| `GRU` | Larga | Media | Alternativa ligera a LSTM |

In [ ]:
# MACHOTE — LSTM para clasificación de texto (sentimientos)

def crear_lstm_texto(vocab_size, max_len, embedding_dim=64, num_clases=2):
    """
    Red LSTM para clasificación de texto.
    
    vocab_size:    tamaño del vocabulario (número de palabras únicas)
    max_len:       longitud máxima de secuencia (después de padding)
    embedding_dim: dimensiones del vector por palabra
    num_clases:    número de categorías de salida
    """
    modelo = keras.Sequential(name="LSTM_Texto")
    
    # Embedding: convierte índices de palabras en vectores densos
    # mask_zero=True → ignora el padding (0s) automáticamente
    modelo.add(layers.Embedding(vocab_size, embedding_dim, mask_zero=True))
    
    # Bidireccional: procesa la secuencia en ambas direcciones → captura más contexto
    modelo.add(layers.Bidirectional(layers.LSTM(64, return_sequences=True)))
    # return_sequences=True → pasa la secuencia completa a la siguiente capa
    
    modelo.add(layers.Bidirectional(layers.LSTM(32)))
    # Sin return_sequences → solo el estado final
    
    modelo.add(layers.Dense(32, activation='relu'))
    modelo.add(layers.Dropout(0.3))
    
    if num_clases == 2:
        modelo.add(layers.Dense(1, activation='sigmoid'))
    else:
        modelo.add(layers.Dense(num_clases, activation='softmax'))
    
    return modelo

# TODO: ajusta según tu corpus
# modelo_lstm = crear_lstm_texto(vocab_size=10000, max_len=200, num_clases=2)
# modelo_lstm.summary()
print("LSTM para texto listo ✓")

In [ ]:
# MACHOTE — Preprocesamiento de texto con Keras Tokenizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

def preprocesar_texto(textos_train, textos_test, vocab_size=10000, max_len=200):
    """
    Convierte listas de strings en secuencias numéricas paddeadas.
    El Tokenizer se entrena SOLO en train para evitar data leakage.
    """
    # Entrenar tokenizador en datos de entrenamiento
    tokenizador = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
    # oov_token → token especial para palabras fuera del vocabulario
    tokenizador.fit_on_texts(textos_train)
    
    # Convertir textos a secuencias de índices
    seq_train = tokenizador.texts_to_sequences(textos_train)
    seq_test  = tokenizador.texts_to_sequences(textos_test)
    
    # Igualar longitudes con padding (rellenar con 0s al inicio)
    X_train_pad = pad_sequences(seq_train, maxlen=max_len, padding='pre', truncating='pre')
    X_test_pad  = pad_sequences(seq_test,  maxlen=max_len, padding='pre', truncating='pre')
    
    print(f"Vocabulario efectivo: {len(tokenizador.word_index)} palabras")
    print(f"Forma train: {X_train_pad.shape} | test: {X_test_pad.shape}")
    
    return X_train_pad, X_test_pad, tokenizador

# Ejemplo de uso:
# X_train_txt, X_test_txt, tok = preprocesar_texto(df_train['texto'], df_test['texto'])
print("Preprocesamiento de texto listo ✓")

In [ ]:
# MACHOTE — LSTM para series de tiempo (regresión)

def crear_series_tiempo(n_pasos, n_features=1, tipo='regresion'):
    """
    Red LSTM para predicción de series de tiempo.
    
    n_pasos:    cuántos pasos de tiempo usa el modelo como contexto
    n_features: número de variables en cada paso de tiempo
    tipo:       'regresion' (predice valor continuo) o 'clasificacion'
    """
    modelo = keras.Sequential(name="LSTM_SeriesTiempo")
    
    # Input shape: (n_pasos, n_features)
    modelo.add(layers.LSTM(64, return_sequences=True, input_shape=(n_pasos, n_features)))
    modelo.add(layers.Dropout(0.2))
    modelo.add(layers.LSTM(32))
    modelo.add(layers.Dropout(0.2))
    modelo.add(layers.Dense(16, activation='relu'))
    
    if tipo == 'regresion':
        modelo.add(layers.Dense(1, activation='linear'))  # sin activación = lineal
    else:
        modelo.add(layers.Dense(2, activation='softmax'))
    
    return modelo

def crear_ventanas_tiempo(serie, n_pasos):
    """
    Convierte una serie 1D en ventanas de tamaño fijo para LSTM.
    Ejemplo: [1,2,3,4,5] con n_pasos=3 → X=[[1,2,3],[2,3,4]], y=[4,5]
    """
    X, y = [], []
    for i in range(len(serie) - n_pasos):
        X.append(serie[i:i+n_pasos])
        y.append(serie[i+n_pasos])
    return np.array(X), np.array(y)

print("LSTM series de tiempo listo ✓")

---
## 📊 5. Evaluación y Diagnóstico de Modelos

In [ ]:
# MACHOTE — Evaluación completa de modelo de clasificación

def evaluar_clasificacion(modelo, X_test, y_test, nombres_clases=None):
    """
    Genera reporte completo: accuracy, matriz de confusión y reporte por clase.
    """
    # Predicciones
    y_pred_prob = modelo.predict(X_test)
    
    if y_pred_prob.shape[-1] == 1:  # binario
        y_pred = (y_pred_prob > 0.5).astype(int).flatten()
    else:  # multiclase
        y_pred = np.argmax(y_pred_prob, axis=1)
    
    # Accuracy global
    acc = np.mean(y_pred == y_test)
    print(f"\n{'='*50}")
    print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    print(f"{'='*50}\n")
    
    # Reporte por clase
    print("Reporte de Clasificación:")
    print(classification_report(y_test, y_pred, target_names=nombres_clases))
    
    # Matriz de confusión
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=nombres_clases, yticklabels=nombres_clases)
    plt.title('Matriz de Confusión')
    plt.ylabel('Real')
    plt.xlabel('Predicho')
    plt.tight_layout()
    plt.show()
    
    return y_pred

# TODO:
# y_pred = evaluar_clasificacion(cnn, X_test_cnn, y_test_raw, 
#                                 nombres_clases=[str(i) for i in range(10)])
print("Evaluación machote listo ✓")

---
## ☁️ 6. Plataformas Comerciales para IA

### ¿Para qué sirven?
Cuando tus recursos locales no son suficientes (o quieres producción), las plataformas cloud ofrecen:
- GPUs/TPUs bajo demanda
- Pipelines de MLOps
- APIs de modelos preentrenados

| Plataforma | Características clave | Free tier |
|---|---|---|
| **Google Colab** | Notebook en la nube con GPU/TPU gratis | Sí |
| **Google Vertex AI** | MLOps completo, AutoML, deploy | $300 créditos |
| **AWS SageMaker** | Entrenamiento managed, endpoints | Capa gratuita |
| **Azure ML** | Integración con Office/Power BI | $200 créditos |
| **Hugging Face** | Modelos NLP/CV preentrenados, Spaces | Sí |
| **Replicate** | API para modelos open source | Pay-per-use |

### Keras → Google Colab (flujo típico)

In [ ]:
# MACHOTE — Guardar y cargar modelos

# Guardar modelo completo (arquitectura + pesos + optimizador)
# modelo.save('mi_modelo.keras')  # formato recomendado en TF2.x

# Cargar modelo guardado
# modelo_cargado = keras.models.load_model('mi_modelo.keras')

# Solo guardar pesos (útil para transfer learning)
# modelo.save_weights('pesos_modelo.weights.h5')
# modelo.load_weights('pesos_modelo.weights.h5')

# Exportar a TensorFlow Lite (para móviles / edge devices)
# converter = tf.lite.TFLiteConverter.from_keras_model(modelo)
# tflite_model = converter.convert()
# with open('modelo.tflite', 'wb') as f:
#     f.write(tflite_model)

print("Machote de guardado/carga listo ✓")

---
## 📋 7. Checklist de Entrega

Antes de entregar un notebook al profe, verifica:

- [ ] El notebook corre completo sin errores (Kernel → Restart & Run All)
- [ ] Los datos están cargados correctamente (no rutas absolutas hardcodeadas)
- [ ] Se muestra `modelo.summary()` con la arquitectura
- [ ] Se grafican las curvas de entrenamiento (loss y accuracy)
- [ ] Se reporta accuracy/métricas en el set de TEST (no solo validación)
- [ ] Se muestra la matriz de confusión
- [ ] Las celdas tienen comentarios explicando el **por qué** de cada paso
- [ ] Las variables tienen nombres descriptivos en español o inglés (no x1, x2, tmp)
- [ ] Si hay OHE de la variable objetivo → target encoding correcto (no OHE de la target)

> 💜 **Tip Diana:** El profe revisa que entiendas el proceso, no solo que funcione.  
> Agrega celdas Markdown explicando qué hace cada bloque y por qué.